In [ ]:
!pip install --no-deps xformers==0.0.27
!pip install --no-deps trl==0.8.6
!pip install --no-deps peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install flask flask-cors nest-asyncio


  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-mw6mt7ox/unsloth_d034ff3ada30473ebf96b6e566b3d952
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-mw6mt7ox/unsloth_d034ff3ada30473ebf96b6e566b3d952
  Resolved https://github.com/unslothai/unsloth.git to commit 93a24f66980b2820083d2882fd3a720d894a1414
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.8.6
    Uninstalling trl-0.8.6:
      Successfully

In [ ]:
import os
from google.colab import files

# --- Configuration ---
# Set the filename of your zip and the folder you want to create
ZIP_FILENAME = "code_fixer_v1-20260331T121910Z-1-001.zip"
EXTRACT_DIR = "code_fixer_v1"
MODEL_PATH = os.path.join(EXTRACT_DIR, "code_fixer_v1")

# 1. Check if model folder exists; if not, ask for upload and unzip
if not os.path.exists(MODEL_PATH):
    if not os.path.exists(ZIP_FILENAME):
        print(f"Please upload {ZIP_FILENAME}")
        uploaded = files.upload()

    print("Unzipping model...")
    import zipfile
    with zipfile.ZipFile(ZIP_FILENAME, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Unzip complete.")
else:
    print("Model already extracted. Proceeding...")

# 2. Load the model using the relative path
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=1024,
    load_in_4bit=True,
)

tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.use_cache = False

print("Model loaded successfully")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.4.4 patched 28 layers with 28 QKV layers, 28 O layers and 0 MLP layers.


Model loaded successfully


In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import torch
import gc
import re
import threading
import traceback
import json

app = Flask(__name__)
CORS(app, resources={r"/*": {"origins": "*"}})

model_lock = threading.Lock()

def extract_json_block(text: str):
    if not text:
        return None

    text = text.strip()
    text = re.sub(r"<\|im_start\|>assistant", "", text, flags=re.IGNORECASE)
    text = re.sub(r"<\|im_end\|>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^assistant[:\s]*", "", text, flags=re.IGNORECASE).strip()

    fenced = re.search(r"```(?:json)?\n?([\s\S]*?)```", text, flags=re.IGNORECASE)
    if fenced:
        text = fenced.group(1).strip()

    try:
        return json.loads(text)
    except:
        pass

    match = re.search(r"\{[\s\S]*\}", text)
    if match:
        try:
            return json.loads(match.group(0))
        except:
            return None

    return None

def clean_code_fallback(text: str) -> str:
    if not text:
        return ""

    text = text.strip()
    text = re.sub(r"<\|im_start\|>assistant", "", text, flags=re.IGNORECASE)
    text = re.sub(r"<\|im_end\|>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^assistant[:\s]*", "", text, flags=re.IGNORECASE).strip()

    fenced = re.search(r"```(?:python)?\n?([\s\S]*?)```", text, flags=re.IGNORECASE)
    if fenced:
        text = fenced.group(1).strip()

    return text.strip()

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"}), 200

@app.route("/analyze_fix", methods=["POST", "OPTIONS"])
def analyze_fix():
    if request.method == "OPTIONS":
        return jsonify({"status": "ok"}), 200

    try:
        data = request.get_json(silent=True) or {}
        user_code = (data.get("code") or "").strip()

        if not user_code:
            return jsonify({"error": "No code provided"}), 400

        prompt = (
            "<|im_start|>system\n"
            "You are an expert Python bug detection and repair engine.\n"
            "Analyze the user's Python code for syntax bugs, logical bugs, semantic bugs, runtime bugs, and edge-case errors.\n"
            "Do not assume the code is correct just because it is syntactically valid.\n"
            "Infer the likely intended behavior from the function name, variable names, conditions, and code structure.\n"
            "If the code is already correct, return has_bug=false, empty buggy_lines, empty detected_bugs, and return the original code unchanged in fixed_code.\n"
            "If the code has bugs, return has_bug=true, list the exact buggy input lines, list the bug types, and return the full corrected Python code.\n"
            "When the bug is logical, identify the incorrect condition, return value, boundary check, or recursion logic.\n"
            "If fixed_code is different from the original input, then has_bug must be true.\n"
            "Return valid JSON only. Do not include markdown. Do not include explanations outside JSON.\n"
            "Use exactly this schema:\n"
            "{\n"
            '  "has_bug": true,\n'
            '  "buggy_lines": ["Line X: <original buggy line content>"],\n'
            '  "detected_bugs": ["<specific bug type>", "<specific bug type>"],\n'
            '  "fixed_code": "<full corrected Python code>"\n'
            "}\n"
            "<|im_end|>\n"
            "<|im_start|>user\n"
            f"{user_code}\n"
            "<|im_end|>\n"
            "<|im_start|>assistant\n"
        )

        with model_lock:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=1024,
                padding=False
            )
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model.generate(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    max_new_tokens=256,
                    do_sample=False,
                    temperature=0.1,
                    use_cache=False,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            input_len = inputs["input_ids"].shape[1]
            generated_tokens = outputs[0][input_len:]
            decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)

        result_data = extract_json_block(decoded)

        if result_data is not None:
            has_bug = bool(result_data.get("has_bug", True))
            fixed_code = (result_data.get("fixed_code") or "").strip()
            buggy_lines = result_data.get("buggy_lines") or []
            detected_bugs = result_data.get("detected_bugs") or []

            if isinstance(buggy_lines, str):
                buggy_lines = [buggy_lines]
            if isinstance(detected_bugs, str):
                detected_bugs = [detected_bugs]

            if not fixed_code:
                fixed_code = user_code

            if fixed_code.strip() != user_code.strip():
                has_bug = True

            if not has_bug:
                buggy_lines = []
                detected_bugs = []

            if has_bug and not buggy_lines:
                buggy_lines = ["AI identified a bug but could not reliably localize the exact line."]
            if has_bug and not detected_bugs:
                detected_bugs = ["Logical / Semantic Bug Detected"]

            return jsonify({
                "has_bug": has_bug,
                "fixed_code": fixed_code,
                "buggy_lines": buggy_lines,
                "detected_bugs": detected_bugs
            }), 200

        fixed_code = clean_code_fallback(decoded)
        has_bug = fixed_code.strip() != user_code.strip()

        return jsonify({
            "has_bug": has_bug,
            "fixed_code": fixed_code if fixed_code else user_code,
            "buggy_lines": ["AI identified a bug but could not reliably localize the exact line."] if has_bug else [],
            "detected_bugs": ["Logical / Semantic Bug Detected"] if has_bug else []
        }), 200

    except Exception as e:
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500


In [ ]:
import socket
import threading
import nest_asyncio

nest_asyncio.apply()

def find_free_port():
    s = socket.socket()
    s.bind(("", 0))
    port = s.getsockname()[1]
    s.close()
    return port

PORT = find_free_port()

def run_app():
    app.run(host="0.0.0.0", port=PORT, debug=False, threaded=False, use_reloader=False)

server_thread = threading.Thread(target=run_app, daemon=True)
server_thread.start()

print(f"Flask server started on port {PORT}")


Flask server started on port 48299


In [ ]:
import os
import subprocess
import time
import requests
from google.colab import userdata # Import this to access secrets

# Retrieve the token securely from your Colab Secrets
try:
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
except:
    raise Exception("Please set the 'NGROK_AUTH_TOKEN' in the Colab Secrets (key icon) on the left sidebar.")

!wget -q -nc https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar -xzf ngrok-v3-stable-linux-amd64.tgz -C /usr/local/bin

# Use the retrieved token here
os.system(f"ngrok config add-authtoken {NGROK_AUTH_TOKEN}")

ngrok_process = subprocess.Popen(
    ["ngrok", "http", str(PORT), "--log=stdout"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(25):
    time.sleep(2)
    try:
        res = requests.get("http://127.0.0.1:4040/api/tunnels", timeout=5)
        data = res.json()
        for tunnel in data.get("tunnels", []):
            url = tunnel.get("public_url", "")
            if url.startswith("https://"):
                public_url = url
                break
        if public_url:
            break
    except:
        pass

if not public_url:
    raise Exception("Ngrok did not start correctly.")

print("Analyze/Fix endpoint:", public_url + "/analyze_fix")
print("Health endpoint:", public_url + "/health")

Analyze/Fix endpoint: https://exportable-juli-nonobligatorily.ngrok-free.dev/analyze_fix
Health endpoint: https://exportable-juli-nonobligatorily.ngrok-free.dev/health


In [ ]:
import time

print("Use this in the website:", public_url + "/analyze_fix")

while True:
    time.sleep(30)
    print("Server alive:", public_url)


Use this in the website: https://exportable-juli-nonobligatorily.ngrok-free.dev/analyze_fix
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev


INFO:werkzeug:127.0.0.1 - - [13/Apr/2026 05:25:48] "OPTIONS /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Apr/2026 05:25:48] "GET /health HTTP/1.1" 200 -


Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev


INFO:werkzeug:127.0.0.1 - - [13/Apr/2026 05:26:11] "OPTIONS /analyze_fix HTTP/1.1" 200 -
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.mask

Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev


...

Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev


INFO:werkzeug:127.0.0.1 - - [13/Apr/2026 05:38:51] "OPTIONS /analyze_fix HTTP/1.1" 200 -
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev


INFO:werkzeug:127.0.0.1 - - [13/Apr/2026 05:40:43] "POST /analyze_fix HTTP/1.1" 200 -


Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-juli-nonobligatorily.ngrok-free.dev
Server alive: https://exportable-j

In [ ]:
import requests
import json

logic_test_cases = [
    {
        "name": "Wrong even/odd logic",
        "code": """def is_even(n):
    return n % 2 == 1"""
    },
    {
        "name": "Wrong factorial base case",
        "code": """def factorial(n):
    if n == 0:
        return 0
    return n * factorial(n - 1)"""
    },
    {
        "name": "Off by one in loop",
        "code": """def sum_n(n):
    total = 0
    for i in range(1, n):
        total += i
    return total"""
    },
    {
        "name": "Wrong maximum selection",
        "code": """def find_max(nums):
    max_num = nums[0]
    for n in nums:
        if n < max_num:
            max_num = n
    return max_num"""
    },
    {
        "name": "Wrong comparison condition",
        "code": """def is_adult(age):
    return age < 18"""
    },
    {
        "name": "Wrong average denominator",
        "code": """def average(nums):
    total = sum(nums)
    return total / (len(nums) - 1)"""
    },
    {
        "name": "Reversed string logic bug",
        "code": """def is_palindrome(s):
    return s == s"""
    },
    {
        "name": "Incorrect list filter condition",
        "code": """def positive_numbers(nums):
    return [n for n in nums if n < 0]"""
    }
]

results = []

for case in logic_test_cases:
    res = requests.post(
        public_url + "/analyze_fix",
        json={"code": case["code"]},
        timeout=180,
        headers={"ngrok-skip-browser-warning": "true"}
    )

    try:
        data = res.json()
    except:
        data = {"error": res.text}

    results.append({
        "name": case["name"],
        "status": res.status_code,
        "response": data
    })

for item in results:
    print("=" * 80)
    print("TEST:", item["name"])
    print("STATUS:", item["status"])
    print(json.dumps(item["response"], indent=2))


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
INFO:werkzeug:127.0.0.1 -

TEST: Wrong even/odd logic
STATUS: 200
{
  "buggy_lines": [
    "return n % 2 == 1"
  ],
  "detected_bugs": [
    "logical bug"
  ],
  "fixed_code": "def is_even(n):\n    return n % 2 == 0",
  "has_bug": true
}
TEST: Wrong factorial base case
STATUS: 200
{
  "buggy_lines": [
    "Line 3: return 0"
  ],
  "detected_bugs": [
    "logical error",
    "runtime error"
  ],
  "fixed_code": "def factorial(n):\n    if n == 0:\n        return 1\n    return n * factorial(n - 1)",
  "has_bug": true
}
TEST: Off by one in loop
STATUS: 200
{
  "buggy_lines": [],
  "detected_bugs": [],
  "fixed_code": "def sum_n(n):\n    total = 0\n    for i in range(1, n+1):\n        total += i\n    return total",
  "has_bug": false
}
TEST: Wrong maximum selection
STATUS: 200
{
  "buggy_lines": [
    "Line 4: if n < max_num:"
  ],
  "detected_bugs": [
    "Logical Bug"
  ],
  "fixed_code": "def find_max(nums):\n    max_num = nums[0]\n    for n in nums:\n        if n > max_num:\n            max_num = n\n    return ma